# Text-to-SQL LoRA fine-tune — Colab runner

Runs the full pipeline on a free Colab **T4 GPU**:

1. install deps + clone the repo
2. `finetune.py --task sql` — LoRA (or QLoRA with `LOAD_IN_4BIT`) on `b-mc2/sql-create-context`
3. `benchmark.py` — base model vs fine-tuned, on a held-out test split
4. optional LLM-as-judge grading (Gemini free tier)
5. download the adapter + results

**Before running:** Runtime → Change runtime type → **T4 GPU**. Total time ≈ 20–30 min
(0.5B model). 7B + QLoRA is ~1–2 h.

In [ ]:
#@title Config
REPO_URL     = "https://github.com/akshatasingh1/llm-fine-tuning.git"  #@param {type:"string"}
BASE_MODEL   = "Qwen/Qwen2.5-0.5B"  #@param ["Qwen/Qwen2.5-0.5B", "Qwen/Qwen2.5-1.5B", "Qwen/Qwen2.5-7B", "HuggingFaceTB/SmolLM2-360M"]
LOAD_IN_4BIT = False  #@param {type:"boolean"}
N_TRAIN      = 4000   #@param {type:"integer"}
N_EVAL       = 200    #@param {type:"integer"}
N_TEST       = 500    #@param {type:"integer"}
EPOCHS       = 2      #@param {type:"integer"}
BATCH_SIZE   = 16     #@param {type:"integer"}
OUTPUT_DIR   = "results/sql_lora"

# 7B on a free T4 needs 4-bit (QLoRA) — flip LOAD_IN_4BIT on and drop BATCH_SIZE to ~4.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → T4 GPU."
print(torch.cuda.get_device_name(0))

In [ ]:
%pip install -q -U "transformers>=5.0.0" "peft>=0.20.0" "accelerate>=1.0.0" "datasets>=3.0.0" "bitsandbytes>=0.43.0" "google-genai>=1.0.0" rouge-score
# Colab ships an old torchao (0.10) that newer peft rejects on import; we don't
# use it, so remove it to avoid the LoRA-injection ImportError.
%pip uninstall -q -y torchao

In [ ]:
import os, pathlib

name = pathlib.Path(REPO_URL).stem
if not os.path.isdir(name):
    !git clone -q "$REPO_URL"
%cd $name
!git pull -q
!ls

## 1. Fine-tune

In [ ]:
import subprocess

cmd = [
    "python", "finetune.py", "--task", "sql",
    "--model_name", BASE_MODEL,
    "--n_train", str(N_TRAIN), "--n_eval", str(N_EVAL), "--n_test", str(N_TEST),
    "--epochs", str(EPOCHS), "--batch_size", str(BATCH_SIZE),
    "--output_dir", OUTPUT_DIR,
]
if LOAD_IN_4BIT:
    cmd.append("--load_in_4bit")
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## 2. Benchmark: base vs fine-tuned

In [ ]:
!python benchmark.py --base_model $BASE_MODEL --adapter_dir $OUTPUT_DIR --n_test $N_TEST

In [ ]:
import json
from IPython.display import Markdown, display

report = json.load(open("results/benchmark.json"))
rows = "\n".join(
    f"| {k} | {report['base'][k]:.1%} | {report['finetuned'][k]:.1%} | {report['delta'][k]:+.1%} |"
    for k in ("exact_match", "execution_match", "valid_sql_rate")
)
display(Markdown(
    f"### {report['base_model']}  ·  n_test = {report['n_test']}\n\n"
    "| metric | base | fine-tuned | Δ |\n|---|---|---|---|\n" + rows
))
display(Markdown(open("results/samples.md").read()))

## 2b. (optional) LLM-as-judge

String/execution metrics miss queries that are correct but written differently.
This grades a sample with Gemini (free tier). Get a key at
https://aistudio.google.com/apikey — or skip this cell.

In [ ]:
import os, getpass, subprocess

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Gemini API key (blank to skip): ")

if os.environ.get("GEMINI_API_KEY"):
    subprocess.run([
        "python", "benchmark.py",
        "--base_model", BASE_MODEL, "--adapter_dir", OUTPUT_DIR,
        "--n_test", str(N_TEST), "--judge", "--judge_n", "100",
    ], check=True)
    import json
    j = json.load(open("results/benchmark.json")).get("judge", {})
    print(json.dumps(j, indent=2))
else:
    print("No key — skipping judge.")

## 3. Download adapter + results

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("sql_lora_run", "zip", "results")
files.download("sql_lora_run.zip")